# CMPE492 — LivePortrait Batch (Stage 3) — CROSS-AUDIO VERSION

**Critical change from previous version:** Both Hybrid-LP AND LP-only outputs are merged with the CROSS-IDENTITY audio (same mapping as Stage 2).

**Why this matters:**
- **Hybrid-LP:** MimicTalk has already produced lips driven by id02's audio (cross). LivePortrait re-renders → merge with id02's audio → lips MATCH audio → high LSE-C expected ✅
- **LP-only:** Raw video has id01's OWN lip motion (saying id01's words). LivePortrait re-renders, but lip motion unchanged → merge with id02's audio → lips DO NOT MATCH audio → low LSE-C expected ✅ (proper control)

**Inputs (BOTH must come from cross-audio Stage 2):**
- `mimic_videos.zip` (from Stage 2 cross-audio) — visuals driven by cross-mapped audio
- `hdtf_prepared.zip` (from Stage 1, unchanged) — raw videos + reference images + audio bank

**Output:** `batch_eval.zip` — ready for Batch Evaluation notebook

---
**Run order:** 1 → 2 → 3 → 4 → 5

## Cell 1 — Setup LivePortrait

In [ ]:
import subprocess, sys, os

gpu = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip()
print('GPU:', gpu)

LP_DIR = '/content/LivePortrait'
if not os.path.exists(LP_DIR):
    subprocess.run('git clone https://github.com/KwaiVGI/LivePortrait.git /content/LivePortrait',
                   shell=True, check=True)
    print('✅ Cloned LivePortrait')
else:
    print('✅ LivePortrait already present')

os.chdir(LP_DIR)

print('Installing dependencies...')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
                   capture_output=True, text=True)
if r.returncode != 0:
    for pkg in ['onnxruntime-gpu', 'opencv-python', 'insightface', 'tyro', 'dill',
                'pykalman', 'imageio', 'imageio-ffmpeg', 'lmdb', 'scikit-image', 'albumentations']:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], capture_output=True)
print('✅ Dependencies installed')

import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
print('\n✅ Cell 1 complete')

## Cell 2 — Download LivePortrait Weights

In [ ]:
import subprocess, os
LP_DIR = '/content/LivePortrait'
os.chdir(LP_DIR)
weights_dir = os.path.join(LP_DIR, 'pretrained_weights')

if not os.path.exists(os.path.join(weights_dir, 'liveportrait', 'base_models', 'warping_module.pth')):
    print('Downloading LivePortrait weights...')
    subprocess.run(
        'huggingface-cli download KwaiVGI/LivePortrait --local-dir pretrained_weights '
        '--exclude "*.git*" "README.md" "docs"',
        shell=True, check=True)
    print('✅ Weights downloaded')
else:
    print('✅ Weights already present')

expected = [
    'liveportrait/base_models/appearance_feature_extractor.pth',
    'liveportrait/base_models/motion_extractor.pth',
    'liveportrait/base_models/spade_generator.pth',
    'liveportrait/base_models/warping_module.pth',
    'liveportrait/retargeting_models/stitching_retargeting_module.pth',
]
missing = [f for f in expected if not os.path.exists(os.path.join(weights_dir, f))]
print(f'{"⚠ Missing: "+str(missing) if missing else "✅ All weights verified"}')

## Cell 3 — Upload Inputs
> Upload BOTH:
> - `mimic_videos.zip` (Stage 2)
> - `hdtf_prepared.zip` (Stage 1)

In [ ]:
import os, zipfile, glob
from google.colab import files

os.makedirs('/content/stage3_inputs/mimic', exist_ok=True)
os.makedirs('/content/stage3_inputs/hdtf', exist_ok=True)

print('Upload mimic_videos.zip AND hdtf_prepared.zip (both at once)')
uploaded = files.upload()
for fname, data in uploaded.items():
    zpath = f'/content/{fname}'
    with open(zpath, 'wb') as f: f.write(data)
    if 'mimic' in fname.lower():
        with zipfile.ZipFile(zpath) as z: z.extractall('/content/stage3_inputs/mimic')
        print(f'✅ {fname} → mimic/')
    elif 'hdtf' in fname.lower() or 'prepared' in fname.lower():
        with zipfile.ZipFile(zpath) as z: z.extractall('/content/stage3_inputs/hdtf')
        print(f'✅ {fname} → hdtf/')

# Discover files
mimic_videos = {}
for v in sorted(glob.glob('/content/stage3_inputs/mimic/**/*.mp4', recursive=True)):
    stem = os.path.splitext(os.path.basename(v))[0]
    mimic_videos[stem] = v

raw_videos = {}
for v in sorted(glob.glob('/content/stage3_inputs/hdtf/**/raw/*.mp4', recursive=True)):
    stem = os.path.splitext(os.path.basename(v))[0]
    raw_videos[stem] = v

references = {}
for img in sorted(glob.glob('/content/stage3_inputs/hdtf/**/references/*.png', recursive=True)):
    stem = os.path.splitext(os.path.basename(img))[0]
    references[stem] = img

print('\n── Discovered ──────────────────────────────')
print(f'  MimicTalk videos: {len(mimic_videos)} → {sorted(mimic_videos.keys())}')
print(f'  Raw videos      : {len(raw_videos)} → {sorted(raw_videos.keys())}')
print(f'  References      : {len(references)} → {sorted(references.keys())}')

# Identities that have everything needed
common = sorted(set(mimic_videos) & set(raw_videos) & set(references))
print(f'\n  ✅ Complete identities (all 3 inputs): {len(common)} → {common}')
missing_any = (set(mimic_videos)|set(raw_videos)|set(references)) - set(common)
if missing_any:
    print(f'  ⚠ Incomplete (will skip): {sorted(missing_any)}')

## Cell 4 — Batch Generate (Hybrid-LP + LP-only, with resume)

In [ ]:
import subprocess, shlex, os, time, glob, shutil, json

LP_DIR = '/content/LivePortrait'
os.chdir(LP_DIR)

OUT_BASE = '/content/batch_eval'
for sub in ['hybrid_lp', 'lp_only', 'references']:
    os.makedirs(f'{OUT_BASE}/{sub}', exist_ok=True)

PROGRESS = '/content/stage3_progress.json'
if os.path.exists(PROGRESS):
    with open(PROGRESS) as f: completed = json.load(f)
else:
    completed = {}

# ── Cross-audio mapping (MUST match Stage 2) ──
def cross_audio_for(stem):
    num = int(stem.replace('id', ''))
    next_num = (num % 10) + 1
    return f'id{next_num:02d}'

def resize_ref(src, dst):
    import cv2
    img = cv2.imread(src)
    cv2.imwrite(dst, cv2.resize(img, (512, 512)))

def run_lp(source_img, driving_vid, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for f in glob.glob(f'{out_dir}/*.mp4'): os.remove(f)
    anim = os.path.join(LP_DIR, 'animations')
    if os.path.exists(anim): shutil.rmtree(anim)
    r = subprocess.run(
        f'python inference.py -s {shlex.quote(source_img)} -d {shlex.quote(driving_vid)} '
        f'--flag_crop_driving_video --driving_smooth_observation_variance 1e-7 '
        f'-o {shlex.quote(out_dir)}', shell=True, capture_output=True, text=True)
    outs = sorted(glob.glob(f'{out_dir}/**/*.mp4', recursive=True))
    if os.path.exists(anim):
        outs += sorted(glob.glob(f'{anim}/**/*.mp4', recursive=True))
    return (outs[0] if outs else None), r

def extract_panel(concat, dst):
    probe = subprocess.run(
        f'ffprobe -v quiet -show_entries stream=width,height -of csv=p=0 "{concat}"',
        shell=True, capture_output=True, text=True).stdout.strip().split(',')
    if len(probe) >= 2:
        w, h = int(probe[0]), int(probe[1])
        if w > h*2:
            pw = w//3
            subprocess.run(
                f'ffmpeg -y -i {shlex.quote(concat)} -vf "crop={pw}:{h}:{pw*2}:0" '
                f'-c:v libx264 -crf 18 {shlex.quote(dst)}', shell=True, capture_output=True)
            return
    shutil.copy(concat, dst)

def merge_audio(gen, audio, dst):
    if audio and os.path.exists(audio):
        subprocess.run(
            f'ffmpeg -y -i {shlex.quote(gen)} -i {shlex.quote(audio)} '
            f'-map 0:v:0 -map 1:a:0 -c:v copy -c:a aac -ac 1 -ar 16000 '
            f'-shortest {shlex.quote(dst)}', shell=True, capture_output=True)
    else:
        shutil.copy(gen, dst)

def smooth_video(src, dst):
    subprocess.run(
        f'ffmpeg -y -i {shlex.quote(src)} -vf "tblend=all_mode=average" '
        f'-c:v libx264 -crf 18 {shlex.quote(dst)}', shell=True, capture_output=True)
    return dst if os.path.exists(dst) else src

# Find the audio source (uploaded with HDTF zip)
def find_audio_file(audio_id):
    cands = glob.glob(f'/content/stage3_inputs/hdtf/**/audio/{audio_id}.wav', recursive=True)
    return cands[0] if cands else None

print('\n── Cross-audio mapping (Hybrid-LP AND LP-only both get cross audio) ──')
for stem in common:
    cross = cross_audio_for(stem)
    print(f'  {stem} visual + {cross} audio')
print()
print(f'Processing {len(common)} identities\n')

for stem in common:
    if stem in completed and \
       os.path.exists(f'{OUT_BASE}/hybrid_lp/{stem}.mp4') and \
       os.path.exists(f'{OUT_BASE}/lp_only/{stem}.mp4'):
        print(f'✓ {stem} already done — skipping')
        continue

    cross_id = cross_audio_for(stem)
    cross_audio_path = find_audio_file(cross_id)
    if not cross_audio_path:
        print(f'⚠ {stem}: cross audio {cross_id} not found, skipping')
        continue

    print(f'\n{"="*55}\n  {stem} ← {cross_id} audio\n{"="*55}')
    t0 = time.time()

    ref_512 = f'/content/stage3_inputs/ref_{stem}_512.png'
    resize_ref(references[stem], ref_512)
    shutil.copy(ref_512, f'{OUT_BASE}/references/{stem}.png')

    # ── Hybrid-LP: smoothed MimicTalk video (already driven by cross_id audio in Stage 2)
    #              → LivePortrait → merge with CROSS audio ──
    print(f'  [1/2] Hybrid-LP (MimicTalk[{cross_id}-driven] → LP, audio={cross_id})...')
    smoothed = smooth_video(mimic_videos[stem], f'/content/stage3_inputs/mimic_sm_{stem}.mp4')
    concat, r = run_lp(ref_512, smoothed, f'/content/lp_tmp/hybrid_{stem}')
    if concat:
        gen = f'/content/lp_tmp/hybrid_{stem}_gen.mp4'
        extract_panel(concat, gen)
        merge_audio(gen, cross_audio_path, f'{OUT_BASE}/hybrid_lp/{stem}.mp4')
        print(f'      ✅ hybrid_lp/{stem}.mp4 (cross audio: {cross_id})')

    # ── LP-only: raw video (still has OWN lip motion) → LP → merge with CROSS audio
    #            → this creates the MISMATCH that proves MimicTalk's necessity ──
    print(f'  [2/2] LP-only (Raw[{stem}-lips] → LP, audio={cross_id} — MISMATCH expected)...')
    concat, r = run_lp(ref_512, raw_videos[stem], f'/content/lp_tmp/baseline_{stem}')
    if concat:
        gen = f'/content/lp_tmp/baseline_{stem}_gen.mp4'
        extract_panel(concat, gen)
        merge_audio(gen, cross_audio_path, f'{OUT_BASE}/lp_only/{stem}.mp4')
        print(f'      ✅ lp_only/{stem}.mp4 (lips don\'t match audio - control)')

    completed[stem] = {'cross_audio_id': cross_id}
    with open(PROGRESS, 'w') as f: json.dump(completed, f, indent=2)
    print(f'  ⏱ {time.time()-t0:.1f}s')

print(f'\n{"="*55}')
print(f'  ✅ Stage 3 (cross-audio) complete: {len(completed)}/{len(common)}')
print(f'{"="*55}')


## Cell 5 — Package for Batch Evaluation

In [ ]:
import shutil, os, glob
from google.colab import files

OUT_BASE = '/content/batch_eval'
for sub in ['hybrid_lp', 'lp_only', 'references']:
    n = len(glob.glob(f'{OUT_BASE}/{sub}/*'))
    print(f'  {sub}: {n} files')

shutil.make_archive('/content/batch_eval', 'zip', OUT_BASE)
sz = os.path.getsize('/content/batch_eval.zip')//(1024*1024)
print(f'\n✅ Packaged: batch_eval.zip ({sz} MB)')
print('  Structure: hybrid_lp/ + lp_only/ + references/')
print('  → Upload this directly to the Batch Evaluation notebook (Cell 2)')

files.download('/content/batch_eval.zip')
print('\n✅ Stage 3 complete. Next: Batch Evaluation notebook')